## 1) Import Libraries

In [9]:
import os
import gower_exp as gower
import numpy as np
import pandas as pd
from config import (
    D_GOWER_PATH,
    OPTIMAL_K_RESULTS_PATH,
    OPTIMAL_K_SUMMARY_PATH,
    OUTPUT_DIR,
    RAW_PATH,
    SCALED_PATH,
    X_PCA_PATH, labels_path
)
from clustering_helpers import (
    load_data,
    run_pca,
    run_kmedoids_gower,
    run_agglomerative_precomputed,
    run_kmeans,
    run_agglomerative_euclidean,
    run_optics,
    compute_prediction_strength
)

## 2) Load Data, Compute Gower, Run PCA


In [10]:

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------
# Load data and prepare feature matrices
# -------------------
print("Loading clustering datasets...")
X_raw, X_scaled, patient_ids = load_data(RAW_PATH, SCALED_PATH)

print("Computing Gower distance for raw data...")
D_gower = gower.gower_matrix(X_raw)

print("Running PCA on scaled data...")
X_pca, var_explained = run_pca(X_scaled)
print(f"PCA reduced to {X_pca.shape[1]} components (explaining {var_explained:.1%} variance)")

Loading clustering datasets...
Computing Gower distance for raw data...
Running PCA on scaled data...
PCA reduced to 2 components (explaining 81.8% variance)


In [11]:
len(X_raw), len(X_scaled), len(X_pca)
X_raw

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=object)

In [12]:
display(X_pca)
print(f"PCA-transformed data shape: {X_pca.shape}")

array([[ 1.21737003, -0.11582529],
       [ 1.21737003, -0.11582529],
       [-0.22035758,  0.08055198],
       ...,
       [-0.22035758,  0.08055198],
       [ 1.21737003, -0.11582529],
       [-0.22035758,  0.08055198]])

PCA-transformed data shape: (1165, 2)


In [13]:
display(D_gower)
print (f"Gower distance matrix shape: {D_gower.shape}")

array([[0.        , 0.        , 0.05263158, ..., 0.05263158, 0.        ,
        0.05263158],
       [0.        , 0.        , 0.05263158, ..., 0.05263158, 0.        ,
        0.05263158],
       [0.05263158, 0.05263158, 0.        , ..., 0.        , 0.05263158,
        0.        ],
       ...,
       [0.05263158, 0.05263158, 0.        , ..., 0.        , 0.05263158,
        0.        ],
       [0.        , 0.        , 0.05263158, ..., 0.05263158, 0.        ,
        0.05263158],
       [0.05263158, 0.05263158, 0.        , ..., 0.        , 0.05263158,
        0.        ]], dtype=float32)

Gower distance matrix shape: (1165, 1165)


## 3) Run Prediction  Strength for each configuration

In [14]:
# -------------------
# Configurations
# -------------------
configs = {
    "raw_kmedoids_gower": (D_gower, run_kmedoids_gower, True),
    "raw_agglomerative_gower": (D_gower, run_agglomerative_precomputed, True),
    
    "pca_kmeans_euclidean": (X_pca, run_kmeans, False),
    "pca_agglomerative_euclidean": (X_pca, run_agglomerative_euclidean, False),
    "pca_optics": (X_pca, run_optics, False), # PS skipped in loop below
}
# -------------------
# Run PS for each config
# -------------------
print("\nStarting prediction strength estimation...")
results = []

for cfg, (X, cluster_fn, precomputed) in configs.items():
    print(f"\nConfiguration: {cfg}")
    if "optics" in cfg:
        labels = cluster_fn(X)
        for k in range(2, 11):
            results.append({"config": cfg, "k": k, "ps": np.nan, "se": np.nan})
        print("   OPTICS skipped (no fixed k).")
        continue

    for k in range(2, 11):
        ps, se = compute_prediction_strength(
            X,
            cluster_fn,
            k,
            precomputed=precomputed,
        )
        
        results.append({"config": cfg, "k": k, "ps": ps, "se": se})
        print(f"   k={k}: PS={ps:.3f}, SE={se:.3f}, PS+SE={ps+se:.3f}")
        # -------------------
# Save results
# -------------------
results_df = pd.DataFrame(results)
results_df.to_csv(OPTIMAL_K_RESULTS_PATH, index=False)
print(f"\nSaved detailed PS results to: OPTIMAL_K_RESULTS_PATH")




Starting prediction strength estimation...

Configuration: raw_kmedoids_gower
   k=2: PS=0.810, SE=0.104, PS+SE=0.914
   k=3: PS=0.994, SE=0.006, PS+SE=0.999
   k=4: PS=0.979, SE=0.008, PS+SE=0.987
   k=5: PS=0.969, SE=0.004, PS+SE=0.973
   k=6: PS=0.984, SE=0.006, PS+SE=0.990
   k=7: PS=1.000, SE=0.000, PS+SE=1.000
   k=8: PS=0.600, SE=0.219, PS+SE=0.819
   k=9: PS=1.000, SE=0.000, PS+SE=1.000
   k=10: PS=1.000, SE=0.000, PS+SE=1.000

Configuration: raw_agglomerative_gower
   k=2: PS=0.980, SE=0.005, PS+SE=0.985
   k=3: PS=0.627, SE=0.126, PS+SE=0.753
   k=4: PS=0.492, SE=0.134, PS+SE=0.626
   k=5: PS=0.581, SE=0.158, PS+SE=0.738
   k=6: PS=0.485, SE=0.190, PS+SE=0.675
   k=7: PS=0.498, SE=0.197, PS+SE=0.695
   k=8: PS=0.590, SE=0.169, PS+SE=0.760
   k=9: PS=0.886, SE=0.102, PS+SE=0.988
   k=10: PS=1.000, SE=0.000, PS+SE=1.000

Configuration: pca_kmeans_euclidean
   k=2: PS=0.799, SE=0.104, PS+SE=0.904
   k=3: PS=0.801, SE=0.109, PS+SE=0.910
   k=4: PS=1.000, SE=0.000, PS+SE=1.000
  

In [15]:
display (results_df)

,config,k,ps,se
0,raw_kmedoids_gower,2,0.809831,0.104160
1,raw_kmedoids_gower,3,0.993799,0.005546
2,raw_kmedoids_gower,4,0.978600,0.008204
3,raw_kmedoids_gower,5,0.968590,0.004323
4,raw_kmedoids_gower,6,0.984004,0.006356
5,raw_kmedoids_gower,7,1.000000,0.000000
6,raw_kmedoids_gower,8,0.600000,0.219089
7,raw_kmedoids_gower,9,1.000000,0.000000
8,raw_kmedoids_gower,10,1.000000,0.000000
9,raw_agglomerative_gower,2,0.980140,0.004997


## 4) Find Optimal K

In [16]:
summary = []
THRESHOLD = 0.8  

for cfg in results_df["config"].unique():
    sub = results_df[results_df["config"] == cfg]
    if "optics" in cfg:
        summary.append({"config": cfg, "k_opt": np.nan, "ps_opt": np.nan, "se_opt": np.nan})
        continue

    # Apply PS + SE >= 0.8 rule
    eligible = sub[(sub["ps"] + sub["se"]) >= THRESHOLD]
    
    if not eligible.empty:
        # Find the largest k satisfying the rule
        best = eligible.loc[eligible["k"].idxmax()]  
    else:
        # Find the most stable k if none meet the threshold
        best = sub.loc[sub["ps"].idxmax()]           

    summary.append({
        "config": cfg,
        "k_opt": int(best["k"]),
        "ps_opt": float(best["ps"]),
        "se_opt": float(best["se"]),
    })
    
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OPTIMAL_K_SUMMARY_PATH, index=False)
print("\nSaved optimal k summary.")


Saved optimal k summary.


## 5) Save Labels

In [17]:
for _, row in summary_df.iterrows():
    cfg = row["config"]
    k = row["k_opt"]

    if pd.isna(k):
        print(f"  Skipping {cfg} (no optimal k)")
        continue

    k = int(k)
    X, cluster_fn, _ = configs[cfg]
    labels = cluster_fn(X, k)

    labels_df = pd.DataFrame({"patient_id": patient_ids, "cluster": labels})
    labels_df.to_csv(labels_path(cfg), index=False, compression="gzip")


  Skipping pca_optics (no optimal k)


In [18]:
display (summary_df)

,config,k_opt,ps_opt,se_opt
0,raw_kmedoids_gower,10.0,1.0,0.0
1,raw_agglomerative_gower,10.0,1.0,0.0
2,pca_kmeans_euclidean,10.0,1.0,0.0
3,pca_agglomerative_euclidean,10.0,1.0,0.0
4,pca_optics,NaN,NaN,NaN


## 6) Save Gower Matrix and PCA data for later use

In [19]:

pd.DataFrame(D_gower).to_csv(D_GOWER_PATH, index=False, compression="gzip")
print(f"Saved Gower distance matrix to: D_GOWER_PATH")

pd.DataFrame(X_pca).to_csv(X_PCA_PATH, index=False, compression="gzip")
print(f"Saved PCA-transformed data to: X_PCA_PATH")

print("Done.")

Saved Gower distance matrix to: D_GOWER_PATH
Saved PCA-transformed data to: X_PCA_PATH
Done.
